# Visualising Data with Matplotlib and Seaborn

Last week you summarised data with numbers. This notebook draws it.

**Part A** walks through each chart type on the male and female population files. **Part B** is for you: you draw charts from the hospital appointments data.

Put these files in the same folder as this notebook:

| File | Used in |
|---|---|
| `sexDensitiesMale.csv`, `sexDensitiesFemale.csv` | Part A and the stretch task |
| `community_health_activity_with_population.csv` | Part B |

---
# Part A: Worked examples



In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:


sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (6, 3.8)

male = pd.read_csv("sexDensitiesMale.csv")
female = pd.read_csv("sexDensitiesFemale.csv")
print(male.shape, female.shape)

## 1. Why draw the data?

Four small datasets (Anscombe's quartet, 1973). The cell below builds them and prints the mean of x, the mean of y, the standard deviation of y and the correlation for each.

In [ ]:
x123 = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = pd.DataFrame({
    "set": ["I"] * 11 + ["II"] * 11 + ["III"] * 11 + ["IV"] * 11,
    "x": x123 * 3 + [8] * 7 + [19] + [8] * 3,
    "y": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68,
          9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74,
          7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73,
          6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
})
for name, g in anscombe.groupby("set"):
    print(name, g["x"].mean().round(2), g["y"].mean().round(2),
          g["y"].std().round(2), g["x"].corr(g["y"]).round(2))

All four rows print the same numbers. Now draw them:

In [ ]:
sns.lmplot(data=anscombe, x="x", y="y", col="set",
           ci=None, height=2.3, aspect=1)
plt.show()

Set I is a scatter around a line. Set II is a curve. Set III is a line with one outlier. Set IV has one point that creates the whole line. The summary numbers cannot see any of that.

## 2. Choose the rows and add columns

The files mix the whole UK, nations, regions, counties and local authorities. A county contains its districts, so using every row would count people twice. Keep local authority rows only (recognised by the first letters of the code), then merge male and female side by side.

In [ ]:
la = ("E06", "E07", "E08", "E09", "W06", "S12", "N09")
m = male[male["CODE"].str.startswith(la)]
f = female[female["CODE"].str.startswith(la)]
df = m.merge(f, on=["CODE", "Location"], suffixes=("_m", "_f"))
print(df.shape)

Four new columns, one for each question we want to chart:

| Column | Meaning |
|---|---|
| `People`, `Thousands` | residents of all ages, and the same number in thousands |
| `Over61` | percentage of residents in the over-61 band |
| `MenPer100` | men for every 100 women aged 19-25 |
| `Nation` | taken from the first letter of the code |

In [ ]:
df["People"] = df["AllAges_m"] + df["AllAges_f"]
df["Thousands"] = df["People"] / 1000
df["Over61"] = (df[">61_m"] + df[">61_f"]) / df["People"] * 100
df["MenPer100"] = df["19-25_m"] / df["19-25_f"] * 100
nations = {"E": "England", "W": "Wales",
           "S": "Scotland", "N": "Northern Ireland"}
df["Nation"] = df["CODE"].str[0].map(nations)
show = ["Location", "Nation", "Thousands", "Over61"]
print(df[show].head(3).round(1))

## 3. Good graphs and common pitfalls

A good chart is **accurate**, **understandable**, **helpful** and **attractive**. The common pitfalls are awkward scales, too much on one chart, bad colour and too much decoration.

### Awkward scales

The same three numbers drawn twice. The only difference is where the y-axis starts.

In [ ]:
three = ["England", "Scotland", "Wales"]
means = df.groupby("Nation")["Over61"].mean()
means = means[three]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].bar(three, means.values)
axes[0].set_ylim(0, 30)
axes[0].set_title("Axis starts at 0")
axes[1].bar(three, means.values)
axes[1].set_ylim(22, 26)
axes[1].set_title("Axis starts at 22")
plt.tight_layout()
plt.show()

Neither chart has a wrong number on it. The right-hand chart misleads through its axis: Wales looks about two and a half times as tall as England, but the real ratio is 25.6 to 23.4, about 1.09. For bar charts, start the axis at zero.

### Bar chart or histogram?

A bar chart compares separate categories, so its bars have gaps. A histogram cuts one number into ranges, so its bars touch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
counts = df["Nation"].value_counts()
axes[0].bar(counts.index, counts.values)
axes[0].set_title("Bar chart: areas per nation")
axes[0].tick_params(axis="x", rotation=15)
axes[1].hist(df["Over61"], bins=20)
axes[1].set_title("Histogram: % of residents over 61")
plt.tight_layout()
plt.show()

### Which chart for which question?

| Chart | Use it to... | Matplotlib / pandas | seaborn |
|---|---|---|---|
| Bar chart | compare a number across categories | `df.plot.bar()` | `sns.barplot()` |
| Histogram | see the shape of one number | `df.plot.hist()` | `sns.histplot()` |
| Box plot | compare spread between groups | `df.plot.box()` | `sns.boxplot()` |
| Scatter plot | see whether two numbers move together | `df.plot.scatter()` | `sns.scatterplot()` |
| Line chart | show change over time | `df.plot.line()` | `sns.lineplot()` |
| Heatmap | compare a number across two categories | `plt.imshow()` | `sns.heatmap()` |

These files have no dates, so we do not draw a line chart today.

## 4. Matplotlib and pandas

The pattern for a pandas plot:

```
dataframe.datafilters.plot.graphtype(options)
plt.show()
```

`dataframe` is the table, `datafilters` is any selection or calculation to get the numbers you want, `graphtype` is `bar`, `barh`, `hist`, `box`, `scatter` or `line`, and `options` are things like `bins=30`. Always label the axes and give the chart a title.

### Histogram: how are the populations spread?

In [ ]:
df.Thousands.plot.hist(bins=30)
plt.xlabel("Residents (thousands)")
plt.title("Local authority populations")
plt.show()

The tail runs far to the right. Which is bigger, the mean or the median? Run the cell to see.

In [ ]:
print(df["Thousands"].mean().round(0))
print(df["Thousands"].median().round(0))

A few very large authorities pull the mean above the median. The median is the better description of a typical area.

### Horizontal bar chart: which ten areas are largest?

`barh` draws the first row at the bottom, so we sort ascending to put the largest at the top.

In [ ]:
top10 = df.nlargest(10, "Thousands")
sizes = top10.set_index("Location")
sizes = sizes["Thousands"].sort_values()
sizes.plot.barh()
plt.xlabel("Residents (thousands)")
plt.title("Ten largest authorities")
plt.show()

## 5. seaborn

The pattern for seaborn: give it the whole table as `data` and name the columns.

```
sns.graphtype(data=df, x="column", y="column", hue="column")
plt.show()
```

`hue` splits the data by a category and gives each group a colour and a legend. seaborn draws with Matplotlib, so `plt.title`, `plt.xlabel` and `plt.show` still work.

### Histogram in seaborn

`kde=True` adds a smooth outline of the shape.

In [ ]:
sns.histplot(data=df, x="Thousands",
             bins=30, kde=True)
plt.xlabel("Residents (thousands)")
plt.title("Local authority populations")
plt.show()

### Box plot: how does the share of older residents vary by nation?

The box covers the middle half of the areas, the line inside is the median, the whiskers reach the furthest values that are not outliers, and the dots are outliers. `order=` sets the order of the boxes.

In [ ]:
order = ["England", "Wales",
         "Scotland", "Northern Ireland"]
sns.boxplot(data=df, x="Nation", y="Over61",
            order=order)
plt.ylabel("% of residents over 61")
plt.title("Older residents by nation")
plt.show()

In [ ]:
df["Nation"].value_counts()

### Scatter plot: is any area unusual?

Each dot is one local authority. The x-axis uses a log scale because sizes range from 2 thousand to over a million. The dashed line is where men and women are equal.

In [ ]:
sns.scatterplot(data=df, x="Thousands",
                y="MenPer100", hue="Nation")
plt.xscale("log")
plt.axhline(100, color="grey", ls="--")
r = df[df["Location"] == "Richmondshire"]
r = r.iloc[0]
plt.text(r["Thousands"] * 0.9,
         r["MenPer100"], "Richmondshire",
         ha="right", va="center")
plt.xlabel("Residents (thousands, log scale)")
plt.ylabel("Men per 100 women aged 19-25")
plt.show()

> **Question.** Richmondshire has about 257 men for every 100 women aged 19-25. What might explain it?

A chart tells you where to look, not why.

## 6. Long data and grouped bars

So far each row was one area, with the age bands as columns (wide data). seaborn wants one row per measurement (long data): one column for the band, one for the number. `melt` turns columns into rows.

In [ ]:
bands = ["<18", "19-25", "26-30", "31-35",
         "35-45", "45-60", ">61"]
cols = dict(id_vars="Location",
            value_vars=bands,
            var_name="Band",
            value_name="People")
long_m = m.melt(**cols)
long_f = f.melt(**cols)
long_m["Sex"] = "Male"
long_f["Sex"] = "Female"
long = pd.concat([long_m, long_f])
print(long.shape)
print(long.head(3))

Now one pair of bars per age band, one bar for each sex:

In [ ]:
pb = long[long["Location"] == "Peterborough"]
sns.barplot(data=pb, x="Band", y="People",
            hue="Sex")
plt.title("Peterborough: people by age band")
plt.show()

Try it: change `"Peterborough"` to your home town and run the cell again.

> **Question.** Peterborough has more boys than girls under 18 but fewer men than women over 61. Is that just Peterborough?

In [ ]:
under18 = df[["<18_m", "<18_f"]].sum()
print((under18["<18_m"] / under18["<18_f"] * 100).round(1))
over61 = df[[">61_m", ">61_f"]].sum()
print((over61[">61_m"] / over61[">61_f"] * 100).round(1))

Across all 390 areas there are about 105 boys per 100 girls under 18 and about 85 men per 100 women over 61.

## 7. Several charts in one figure

`plt.subplots(rows, columns)` creates a grid of axes. Draw each chart into one of them with `ax=`. With one row, use `axes[0]`, `axes[1]`, `axes[2]`; with several rows use `axes[row, column]`. `axes.flatten()` turns a grid into a list, and `fig.delaxes(axes[1, 2])` removes an unused spot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

df["Thousands"].plot.hist(bins=30, ax=axes[0])
sns.boxplot(data=df, x="Nation", y="Over61",
            order=order, ax=axes[1])
sns.scatterplot(data=df, x="Thousands",
                y="MenPer100", hue="Nation",
                ax=axes[2])

axes[0].set_title("Population (thousands)")
axes[1].set_title("% of residents over 61")
axes[2].set_title("Men per 100 women, 19-25")
axes[1].tick_params(axis="x", rotation=20)
axes[2].set_xscale("log")
plt.tight_layout()
plt.show()

## 8. Style and formatting

- Change one chart: options inside the call, such as `color="navy"`, `linestyle="dotted"`, `marker="o"`.
- Change every chart: `plt.style.use("ggplot")`.
- seaborn: `sns.set_theme(style=..., palette=...)`.

The same bar chart in three built-in styles. `plt.style.context` applies a style only inside the `with` block. `plt.style.available` lists every style.

In [ ]:
def draw(style):
    with plt.style.context(style):
        sizes.plot.barh(title=style)
        plt.show()

for name in ["ggplot", "bmh", "dark_background"]:
    draw(name)

---
# Part B: Your turn

You draw the charts from here on.

**Rules for Part B**

- You may use AI to help write plotting code, but try one plot first using the reference pages.
- You must be able to explain what every part of your code does, with no AI on screen.
- Do not accept AI offers to restyle or add extras. Draw only what each task asks.

**Where to look first**

- Matplotlib plot types: https://matplotlib.org/stable/plot_types/index.html
- seaborn example gallery: https://seaborn.pydata.org/examples/index.html
- seaborn function reference: https://seaborn.pydata.org/api.html
- pandas visualisation guide: https://pandas.pydata.org/docs/user_guide/visualization.html

Find the chart, copy the example, change the column names.

## The hospital data

Twelve rows: appointments in three areas, with population and deprivation band added. One row is one area, one year, one age group and one sex. One appointments value and one population are missing, so `appointments_per_1000` is missing for that row too.

In [ ]:
health = pd.read_csv("community_health_activity_with_population.csv")
print(health.shape)
print(health.isna().sum()[["appointments", "population"]])
print(health[["area_name", "age_group", "sex"]].head(4))

## Task 1: Appointments by area

Draw a bar chart of the total appointments in each area using Matplotlib (pandas `.plot`).

1. Group `health` by `area_name` and add up `appointments`. Print the totals.
2. Draw the totals as a bar chart with axis labels and a title.

In [ ]:
# Your code here
# groupby area_name, sum appointments, print, then .plot.bar()

## Task 2: Wait by deprivation band

Use seaborn to draw the average wait in days for each deprivation band.

1. Print the averages with `groupby` first, so you know what the bars should show.
2. Use `sns.barplot` with the bands in the order Low, Medium, High.
3. Label the axes and add a title.

In [ ]:
# Your code here
# groupby deprivation_band mean, then sns.barplot(..., order=[...])

## Task 3: Rate and wait

Draw a scatter plot of `appointments_per_1000` against `average_wait_days`, coloured by `age_group`. Choose seaborn or Matplotlib and use the reference pages to find the example. Label both axes with units and add a title. Count the dots on your chart.

In [ ]:
# Your code here
# scatter of appointments_per_1000 against average_wait_days, coloured by age_group

> **Question.** How many dots should there be if the table has 12 rows? Which age group has the lowest rates, and what do you notice about its waits?


## Task 4: One figure

Put your three charts into one figure side by side. Use `plt.subplots(1, 3)`, draw each chart into its own axes with `ax=`, give each a title and finish with `plt.tight_layout()`. Choose a style and say why it is readable.

In [ ]:
# Your code here
# fig, axes = plt.subplots(1, 3, ...) then draw each chart with ax=axes[i]

> **Question.** If this went in a report for managers, which of your three charts would you keep, and why?

*Your answer:*

## Stretch: your own town

Go back to the male and female files and draw the age-band chart for a place of your choice, as for Peterborough. Write one sentence comparing your town with Peterborough. Which age band shows the biggest difference?

In [ ]:
# Your code here
# filter long to your town, then sns.barplot(..., hue="Sex")

*Your sentence:*

## Experiment

See what other useful plot you can make with these datasets.

Apply different styling to your plots.

Pick up any Keggle(https://www.kaggle.com/) dataset and apply different visulization on it